In [1]:
!nvidia-smi

Thu Oct  1 06:57:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!git clone https://github.com/hannna24/Github-IssueHawk.git

Cloning into 'Github-IssueHawk'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 127 (delta 47), reused 113 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.54 MiB | 4.79 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [3]:
%cd Github-IssueHawk

/content/Github-IssueHawk


In [4]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 71.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 59.9 MB/s eta 0:00:00


## A. Set location & imports
Make sure we're in the `training/` folder (so `from prepare_dataset import ...` works) and load every library we need.



In [6]:
%cd /content/Github-IssueHawk/training

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from prepare_dataset import build_dataset

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
print('imports OK')

/content/Github-IssueHawk/training
imports OK


In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass('Hugging Face token (write access): '))

## B. Detect GPU precision (the T4 fix)
A T4 has no bf16 hardware, so this auto-picks **fp16**. You should see `using fp16 (correct for a T4)`.



In [8]:
BF16_OK = torch.cuda.get_device_capability(0)[0] >= 8   # native bf16 needs Ampere+; T4 is 7.5
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'using {"bf16" if BF16_OK else "fp16"} (correct for a T4)')

GPU: Tesla T4
using fp16 (correct for a T4)


## C. Load the base model in 4-bit
Downloads Qwen2.5-3B (~a few GB, **one-time**) and loads it in 4-bit so it fits the T4.
Slowest cell the first time — a few minutes. Ignore any `HF_TOKEN` warning; it's harmless.


In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,   # float16 on a T4
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto'
)
print('\n✅ base model loaded in 4-bit')

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ base model loaded in 4-bit


## D. Attach the LoRA adapters
Freezes the base model and injects small trainable adapters into the attention layers.
The printout should show **trainable params well under 1%** — that's LoRA working.


In [10]:
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()   # essential on a free GPU

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)

# fp16 training (GradScaler) can't unscale bf16 grads; Qwen loads as bf16 by default,
# so force every trainable param to fp32.
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

print('trainable dtypes:', {p.dtype for p in model.parameters() if p.requires_grad})
model.print_trainable_parameters()

trainable dtypes: {torch.float32}
trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


## E. Build the train & validation datasets
Formats your real Layer 1 splits into chat-template examples. You saw this work already (1639 rows).


In [11]:
train_ds = build_dataset('../data/splits/train.jsonl')
val_ds   = build_dataset('../data/splits/val.jsonl')
print(f'train: {len(train_ds)} examples   val: {len(val_ds)} examples')

train: 1639 examples   val: 351 examples


## F. Configure the trainer (T4-safe)
All the training settings. The `try/except` blocks handle newer-vs-older versions of the `trl`
library automatically (the renamed `max_length` and `processing_class`), so you won't hit a `TypeError`.


In [12]:
import os

# Save to the Colab runtime disk (Drive-mount doesn't play nice with the VS Code
# Colab extension — MessageError: [dfs_ephemeral] Credentials propagation unsuccessful).
# Checkpoints are also pushed to the HF Hub so a runtime disconnect doesn't lose them.
CKPT_DIR = '../checkpoints/lora-adapter'
os.makedirs(CKPT_DIR, exist_ok=True)
print('will save adapter to:', CKPT_DIR)

cfg = dict(
    output_dir=CKPT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,     # T4-safe (doc used 4)
    gradient_accumulation_steps=8,     # 2 x 8 = effective batch 16
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy='steps', eval_steps=50,
    save_strategy='steps',             # save mid-epoch, not just at epoch boundaries
    save_steps=25,                     # ~4 saves per epoch (survives same-runtime crashes)
    save_total_limit=3,                # keep only the 3 newest to bound disk usage
    bf16=BF16_OK, fp16=not BF16_OK,    # fp16 on a T4
    report_to='none',                  # no wandb login needed
    dataset_text_field='text',
    loss_type='nll',                   # trl's default chunked_nll crashes with PEFT + 4-bit
    push_to_hub=True,
    hub_model_id='hanabakeer24/issuehawk-qwen2.5-3b-lora',  # change to your HF username/repo
    hub_strategy='checkpoint',         # push every checkpoint to the last-checkpoint branch
    hub_private_repo=True,
)
try:
    sft_config = SFTConfig(max_length=1024, **cfg)      # newer trl
except TypeError:
    sft_config = SFTConfig(max_seq_length=1024, **cfg)  # older trl

try:
    trainer = SFTTrainer(model=model, args=sft_config,
                         train_dataset=train_ds, eval_dataset=val_ds,
                         processing_class=tokenizer)     # newer trl
except TypeError:
    trainer = SFTTrainer(model=model, args=sft_config,
                         train_dataset=train_ds, eval_dataset=val_ds,
                         tokenizer=tokenizer)            # older trl
print('✅ trainer ready')

will save adapter to: ../checkpoints/lora-adapter


Adding EOS to train dataset:   0%|          | 0/1639 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1639 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1639 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1639 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/351 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/351 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/351 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/351 [00:00<?, ? examples/s]

✅ trainer ready


In [13]:
# The trainer can re-cast adapter weights to the base model's bf16; fp16 training needs fp32 trainables.
print('before:', {p.dtype for p in trainer.model.parameters() if p.requires_grad})
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
print('after: ', {p.dtype for p in trainer.model.parameters() if p.requires_grad})

before: {torch.bfloat16}
after:  {torch.float32}


## G. Train  ▶️  *(the long one — ~30–90 min)*
A progress bar appears and loss values print as it goes. **A steadily falling loss is healthy.**

Checkpoints save every 25 steps to the Colab runtime disk and are pushed to your Hub repo (`last-checkpoint/`). If your kernel crashes but the runtime is still alive, just re-run the training cell — it auto-resumes. If Colab **disconnects** the whole runtime, the disk is wiped: start a new runtime, re-run the setup cells (through F and the dtype cell), run the restore cell below, then the training cell. When training finishes, upload the adapter in cell H.

In [14]:
# Run ONLY after a Colab disconnect: restore the latest checkpoint from the Hub, then run the next cell.
# (hub_strategy='checkpoint' stores it in a `last-checkpoint/` folder inside the repo.)
from huggingface_hub import snapshot_download
import os
os.makedirs(CKPT_DIR, exist_ok=True)
snapshot_download(
    repo_id='hanabakeer24/issuehawk-qwen2.5-3b-lora',
    allow_patterns=['last-checkpoint/*'],
    local_dir=CKPT_DIR,
)
print('restored:', sorted(os.listdir(CKPT_DIR)))
print('inside last-checkpoint:', sorted(os.listdir(f'{CKPT_DIR}/last-checkpoint')))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

restored: ['.cache', 'last-checkpoint']
inside last-checkpoint: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'optimizer.pt', 'rng_state.pth', 'scaler.pt', 'scheduler.pt', 'tokenizer.json', 'tokenizer_config.json', 'trainer_state.json', 'training_args.bin']


In [15]:
import os
# Resume priority: a checkpoint restored from the Hub (last-checkpoint/), else a local checkpoint-N, else fresh.
hub_ckpt = os.path.join(CKPT_DIR, 'last-checkpoint')
local_ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]
if os.path.isdir(hub_ckpt):
    resume = hub_ckpt
elif local_ckpts:
    resume = True
else:
    resume = False
print(f'{"▶️  Resuming from " + str(resume) if resume else "🆕 Starting fresh"} — output dir: {CKPT_DIR}')

trainer.train(resume_from_checkpoint=resume)
trainer.save_model(CKPT_DIR)
print(f'\n✅ Adapter saved to {CKPT_DIR}')

▶️  Resuming from ../checkpoints/lora-adapter/last-checkpoint — output dir: ../checkpoints/lora-adapter


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
300,0.846544,0.844179,0.833070,0.812254,237037.000000
309,0.846544,0.844127,0.832770,0.812079,276958.000000



✅ Adapter saved to ../checkpoints/lora-adapter


## H. Upload the final adapter to the Hugging Face Hub *(do this before Colab disconnects!)*
The adapter lives on the Colab runtime's temporary disk, and a disconnect wipes it. This uploads only the finished adapter (~30 MB, no optimizer or checkpoint files) to the `final-adapter/` folder of your Hub repo. `merge_adapter.py` loads it from there.

In [18]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=CKPT_DIR,
    repo_id='hanabakeer24/issuehawk-qwen2.5-3b-lora',
    path_in_repo='final-adapter',
    ignore_patterns=['checkpoint-*', 'last-checkpoint*', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'scaler.pt'],
)
print('uploaded')


uploaded
